In [106]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import numpy as np
import pandas as pd
from src.utils import read_csv_file
import config
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from pytabkit import TabM_D_Classifier
import torch

In [107]:
train = read_csv_file('data/raw/train.csv')
X = train.drop(['id', 'irrigation_need'], axis=1)
y = train['irrigation_need'].map(config.TARGET_MAP)

Reading: data/raw/train.csv


In [108]:
from sklearn.model_selection import train_test_split

# X = features, y = target
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(504000, 19) (126000, 19)


In [109]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
oof_preds = np.zeros(len(X), dtype=int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n--- Fold {fold} ---")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = TabM_D_Classifier(
        device='cuda',
        random_state=42,
        n_cv=1,
        n_refit=0,
        n_repeats=1,
        val_fraction=0.0,
        n_threads=12,
        tmp_folder=None,
        verbosity=1,

        # architecture
        arch_type='tabm-mini',
        tabm_k=32,
        num_emb_type='pwl',
        num_emb_n_bins=48,

        # training
        batch_size=512,
        lr=None,
        weight_decay=None,
        n_epochs=32,
        patience=16,

        # model size
        d_embedding=None,
        d_block=128,
        n_blocks=6,
        dropout=None,

        # advanced
        compile_model=False,
        allow_amp=True,
        tfms=None,
        gradient_clipping_norm=None,

        # metrics / calibration
        share_training_batches=False,
        val_metric_name='1-balanced_accuracy',
        train_metric_name=None
    )

    model.fit(X_train, y_train, X_val, y_val, cat_col_names=cat_cols)
    fold_pred = model.predict(X_val)

    oof_preds[val_idx] = fold_pred

    score = balanced_accuracy_score(y_val, fold_pred)
    print(f"Fold {fold} Score: {score:.4f}")

print(f'\nOOF Balanced accuracy: {balanced_accuracy_score(y, oof_preds):.4f}')


--- Fold 1 ---
Columns classified as continuous: ['soil_ph', 'soil_moisture', 'organic_carbon', 'electrical_conductivity', 'temperature_c', 'humidity', 'rainfall_mm', 'sunlight_hours', 'wind_speed_kmh', 'field_area_hectare', 'previous_irrigation_mm']
Columns classified as categorical: ['soil_type', 'crop_type', 'crop_growth_stage', 'season', 'irrigation_type', 'water_source', 'mulching_used', 'region']
Device:        CUDA
AMP:           True (dtype: torch.bfloat16)
torch.compile: False
----------------------------------------------------------------------------------------



Epoch 0: 100%|██████████| 985/985 [00:08<00:00, 122.65it/s]


(val) -0.0396
🌸 New best epoch! 🌸



Epoch 1: 100%|██████████| 985/985 [00:08<00:00, 121.64it/s]


(val) -0.0384
🌸 New best epoch! 🌸



...

(val) -0.0381



Epoch 30: 100%|██████████| 985/985 [00:21<00:00, 46.81it/s]


(val) -0.0387



Epoch 31: 100%|██████████| 985/985 [00:21<00:00, 45.68it/s]


(val) -0.0402


Result:
{'val': -0.034449100494384766, 'epoch': 14}
Restoring best model
Fold 2 Score: 0.9655

--- Fold 3 ---
Columns classified as continuous: ['soil_ph', 'soil_moisture', 'organic_carbon', 'electrical_conductivity', 'temperature_c', 'humidity', 'rainfall_mm', 'sunlight_hours', 'wind_speed_kmh', 'field_area_hectare', 'previous_irrigation_mm']
Columns classified as categorical: ['soil_type', 'crop_type', 'crop_growth_stage', 'season', 'irrigation_type', 'water_source', 'mulching_used', 'region']
